# Medical QSVM: understanding the CPU and MerLin adaptations

This notebook presents the small results that have already been computed. It does not rerun a quantum simulation or kernel calculation.

Three scopes must remain distinct:

1. the reference reproduction using the paper's controlled medical embeddings, which is not performed here;
2. the surrogate study using PneumoniaMNIST pixels;
3. the MerLin photonic adaptation, which is not the paper's qubit BSP circuit.

## 1. Load the curated artifacts

The files under `results/` contain only small metrics and metadata. Images, pixels, and kernel matrices are not included.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
results_path = project_root / "results"
summary_path = results_path / "curated_results.csv"
merlin_path = results_path / "merlin_q2_seed0.json"

if not summary_path.exists() or not merlin_path.exists():
    raise FileNotFoundError("Run this notebook from the qsvm_medimage root.")

results = pd.read_csv(summary_path)
results

## 2. CPU comparison at q=4

The primary metric is the **minority-class F1** (`normal`). It combines:

- precision: among images predicted as normal, how many are actually normal;
- recall: among genuinely normal images, how many are detected.

A minority-class F1 of zero means that no minority example was correctly detected.

In [ ]:
q4 = results.query("scope == 'open_data_surrogate' and pca_dim == 4").copy()
q4[["model", "n_seeds", "mean_test_minority_f1", "std_test_minority_f1"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    q4["model"],
    q4["mean_test_minority_f1"],
    yerr=q4["std_test_minority_f1"],
    capsize=4,
)
ax.set_ylabel("Test minority-class F1")
ax.set_ylim(0, 1.05)
ax.set_title("PneumoniaMNIST, q=4, N=100, 10 paired seeds")
ax.tick_params(axis="x", rotation=15)
fig.tight_layout()

The QSVM mean is slightly above the linear SVM (`+0.013`) but slightly below the tuned RBF (`-0.020`). These differences are small compared with the variability across seeds. This study therefore does not reproduce the systematic advantage reported in the paper.

## 3. MerLin photonic-kernel smoke

This experiment uses two PCA components, three optical modes, and the Fock state `[1, 0, 1]`. The data seed and circuit seed are both zero, but they have distinct roles.

In [ ]:
with merlin_path.open() as file:
    merlin = json.load(file)

merlin_metrics = pd.DataFrame(merlin["metrics"]).T
merlin_metrics.index.name = "split"
merlin_metrics

In [ ]:
ax = merlin_metrics[["accuracy", "minority_f1", "auc"]].plot.bar(
    figsize=(8, 4), rot=0
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("MerLin: PCA=2, modes=3, data seed=0, circuit seed=0")
ax.figure.tight_layout()

The test AUC is 1, while the minority-class F1 is 0. This is not contradictory: AUC measures score ranking across all possible thresholds, whereas F1 uses the decisions produced at the SVC threshold. Here, the model ranks the scores correctly but ultimately predicts every image as `pneumonia`.

The test split contains only ten images, including two normal images. This result validates that MerLin runs technically on CPU; it does not demonstrate a photonic advantage.

## 4. What remains to be established

- PneumoniaMNIST pixels and the paper's frozen embeddings are not scientifically equivalent.
- The MerLin kernel and the qubit BSP circuit are different feature maps.
- The current results validate the pipeline and guide subsequent experiments; they do not support a quantum-advantage claim.
- Every subsequent comparison must use the same samples and splits for all compared models.